# Experimento 1 — Random Forest sem balanceamento utilizando 4 variáveis

O primeiro experimento teve como objetivo criar uma baseline inicial do comportamento do modelo Random Forest antes da aplicação de qualquer técnica de balanceamento.

A ideia principal desse primeiro teste foi observar como o algoritmo se comporta utilizando um conjunto reduzido de variáveis ambientais e mantendo a distribuição original do dataset.

As variáveis utilizadas como entrada foram:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `Country`
- `Waterbody Type`

Enquanto isso, a variável alvo (`y`) utilizada para previsão foi:

- `conama_status`

Nesse experimento, o modelo não recebeu diretamente os parâmetros utilizados na construção do rótulo, como pH, DBO, oxigênio dissolvido, nitrato e amônia. O objetivo disso foi justamente verificar se o algoritmo conseguiria encontrar relações indiretas entre outras variáveis ambientais e as classificações definidas pelo `conama_status`.

Essa abordagem é importante porque, em cenários reais, muitos parâmetros ambientais possuem relações indiretas entre si. Mesmo que uma variável não tenha sido utilizada diretamente na construção do rótulo, ela ainda pode carregar informações relevantes sobre o comportamento ambiental da água.

O dataset foi dividido em:

- 80% para treino
- 20% para teste

A separação foi realizada utilizando `train_test_split` com `random_state=42`, garantindo reprodutibilidade experimental.

Além disso, foi utilizado `stratify=y`, permitindo manter a proporção original das classes nos conjuntos de treino e teste. Isso foi importante porque o dataset apresenta desbalanceamento significativo entre as classificações.

Durante o treinamento, o Random Forest tentou aprender padrões relacionando as variáveis de entrada com as classificações do `conama_status`.

Após o treinamento, o modelo foi avaliado utilizando:

- Accuracy
- Precision
- Recall
- F1-score
- Matriz de confusão

O principal objetivo desse primeiro experimento não foi obter o melhor resultado possível, mas criar um ponto de comparação inicial para os próximos testes.

A partir dele, foi possível observar:

- impacto do desbalanceamento;
- tendência do modelo favorecer classes majoritárias;
- dificuldade na identificação de classes críticas;
- diferença entre métricas de treino e teste;
- primeiros sinais de overfitting.

Esses resultados servirão como base para comparar os próximos experimentos, principalmente aqueles envolvendo balanceamento e inclusão de novas variáveis ambientais.

## Preparação do ambiente


In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [ ]:
# preparando o ambiente para machine learning
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [ ]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (113119, 4)
Teste: (28280, 4)


In [ ]:
# pré-processamento

categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            RandomForestClassifier(
                random_state=SEED
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.8843695577223986
Train Precision:
0.880890036033501
Train Recall:
0.8843695577223986
Train F1:
0.8786477536700549
Train Confusion Matrix:
[[ 4654  1357    28  1521]
 [  248 14377    27  7026]
 [   79   197   703   127]
 [  178  2268    24 80305]]


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Teste com 4 variáveis - sem balanceamento
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7103253182461103

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.32      0.20      0.24      1890
         Boa       0.33      0.24      0.28      5419
     Crítica       0.10      0.05      0.06       277
   Excelente       0.80      0.89      0.84     20694

    accuracy                           0.71     28280
   macro avg       0.39      0.34      0.36     28280
weighted avg       0.67      0.71      0.69     28280


Confusion Matrix:
[[  370   668    35   817]
 [  369  1304    46  3700]
 [   63   103    13    98]
 [  362  1894    37 18401]]


## Resultados Obtidos — Experimento 1

Após o treinamento do modelo Random Forest utilizando apenas as quatro variáveis iniciais e sem aplicação de balanceamento, foi possível observar alguns comportamentos importantes do modelo.

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.71
```

Enquanto isso, a acurácia no conjunto de treino ficou em torno de:
```python
0.88
```
Essa diferença entre treino e teste já indica sinais de overfitting.

Isso significa que o modelo conseguiu aprender muito bem os padrões presentes no conjunto de treino, mas apresentou uma redução considerável de desempenho ao tentar generalizar para dados novos do conjunto de teste.

Além disso, ao analisar as métricas individuais das classes, foi possível perceber que o modelo apresentou desempenho muito superior para a classe `Excelente`, enquanto teve bastante dificuldade para identificar corretamente as classes minoritárias, principalmente `Crítica`.

Esse comportamento já era esperado devido ao forte desbalanceamento presente no dataset.

A matriz de confusão mostrou que o modelo possui tendência de classificar muitas amostras como `Excelente`, mesmo quando elas pertencem a outras categorias.

Isso acontece porque a classe `Excelente` representa a maior parte do dataset, fazendo com que o modelo aprenda mais facilmente os padrões associados a ela.

Como consequência:

* classes minoritárias tiveram baixo recall;
* muitas amostras críticas foram classificadas incorretamente;
* o modelo apresentou dificuldade em separar classes intermediárias;
* houve tendência de favorecimento da classe majoritária.

Mesmo assim, o experimento foi extremamente importante como baseline inicial da análise experimental.

A partir dele, tornou-se possível:

* identificar sinais de overfitting;
* observar o impacto do desbalanceamento;
*entender limitações iniciais do modelo;
* criar uma referência para comparação com os próximos experimentos.

Os próximos testes buscarão reduzir esses problemas utilizando:

* técnicas de balanceamento;
* novas variáveis;
* diferentes configurações de entrada;
* comparação entre algoritmos.


